Random forest classifier model to calculate the performance score of an employee

Before Building the model, let's analyse the dataset and see if it is fit or not

In [1]:
import pandas as pd
df=pd.read_csv("Employee_Performance.csv")
df.shape

(100000, 20)

So we have 1lakh entries with 20 columns

Let's see the columns we have, we have to make sure that the model should be generic

In [2]:
df.columns.tolist()

['Employee_ID',
 'Department',
 'Gender',
 'Age',
 'Job_Title',
 'Hire_Date',
 'Years_At_Company',
 'Education_Level',
 'Performance_Score',
 'Monthly_Salary',
 'Work_Hours_Per_Week',
 'Projects_Handled',
 'Overtime_Hours',
 'Sick_Days',
 'Remote_Work_Frequency',
 'Team_Size',
 'Training_Hours',
 'Promotions',
 'Employee_Satisfaction_Score',
 'Resigned']

In [3]:
df.drop(
    ["Employee_ID",
     "Department",
     "Gender",
     "Job_Title",
     "Hire_Date",
     "Years_At_Company",
     "Education_Level",
     "Team_Size",
     "Promotions",
     "Resigned",
     "Remote_Work_Frequency",
     "Monthly_Salary",
     "Age"
     ],axis=1,inplace=True

)

In [4]:
df.columns.tolist()

['Performance_Score',
 'Work_Hours_Per_Week',
 'Projects_Handled',
 'Overtime_Hours',
 'Sick_Days',
 'Training_Hours',
 'Employee_Satisfaction_Score']

In [5]:
df.shape

(100000, 7)

Now our goal is to build model to classify the performance score, before that let us see type of classifications that are done for performance score

In [6]:
print(df["Performance_Score"].value_counts())
print("-"*100)
print(df["Performance_Score"].value_counts(normalize=True))

Performance_Score
1    20120
2    20013
3    19999
4    19940
5    19928
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
Performance_Score
1    0.20120
2    0.20013
3    0.19999
4    0.19940
5    0.19928
Name: proportion, dtype: float64


Let's check if there are any null values before moving forward

In [7]:
print(df.isnull().sum())

Performance_Score              0
Work_Hours_Per_Week            0
Projects_Handled               0
Overtime_Hours                 0
Sick_Days                      0
Training_Hours                 0
Employee_Satisfaction_Score    0
dtype: int64


There is one null value in each column, let's drop the null value and make the dataset perfect

In [8]:
df.dropna(inplace=True)

Let's verify it

In [9]:
df.isnull().sum()

,0
Performance_Score,0
Work_Hours_Per_Week,0
Projects_Handled,0
Overtime_Hours,0
Sick_Days,0
Training_Hours,0
Employee_Satisfaction_Score,0


In [10]:
df.shape

(100000, 7)

Now let's see a summary about the values in the dataframe

In [11]:
df.describe().round(2)

,Performance_Score,Work_Hours_Per_Week,Projects_Handled,Overtime_Hours,Sick_Days,Training_Hours,Employee_Satisfaction_Score
count,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00
mean,3.00,44.96,24.43,14.51,7.01,49.51,3.00
std,1.41,8.94,14.47,8.66,4.33,28.89,1.15
min,1.00,30.00,0.00,0.00,0.00,0.00,1.00
25%,2.00,37.00,12.00,7.00,3.00,25.00,2.01
50%,3.00,45.00,24.00,15.00,7.00,49.00,3.00
75%,4.00,53.00,37.00,22.00,11.00,75.00,3.99
max,5.00,60.00,49.00,29.00,14.00,99.00,5.00


Almost everything seems to be in a reasonable range and no outliers seen which is a good sign to proceed

Now let's start splitting the dataset

In [12]:
X=df.drop(["Performance_Score"],axis=1)
y=df["Performance_Score"]
print(X.shape)
print(y.shape)

(100000, 6)
(100000,)


Now let's start splitting the dataset into 80% training and 20% testing dataset

In [13]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(80000, 6)
(20000, 6)
(80000,)
(20000,)


We have used stratify=y to make sure that the ratios are splitted properly so that the model would learn about everything, let's check if the split is done properly

In [14]:
print("Train distribution")
print(y_train.value_counts(normalize=True).round(3) * 100)
print("Test distribution")
print(y_test.value_counts(normalize=True).round(3)*100)

Train distribution
Performance_Score
1    20.1
2    20.0
3    20.0
4    19.9
5    19.9
Name: proportion, dtype: float64
Test distribution
Performance_Score
1    20.1
2    20.0
3    20.0
4    19.9
5    19.9
Name: proportion, dtype: float64


Okay the data is evenly distributed, let's start building the model

In [15]:
from sklearn.ensemble import RandomForestClassifier
rf_model=RandomForestClassifier(
     n_estimators=100,
     max_depth=10,
     min_samples_split=10,
     min_samples_leaf=4,
     class_weight="balanced",
     random_state=42,
     n_jobs=-1
 )
rf_model.fit(X_train,y_train)
print("Model is successfully trained")

Model is successfully trained


Now let's evaluate the model and see how well it has been trained

In [19]:
from sklearn.metrics import classification_report,confusion_matrix
y_pred=rf_model.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           1       0.20      0.20      0.20      4024
           2       0.19      0.19      0.19      4003
           3       0.20      0.21      0.21      4000
           4       0.19      0.23      0.21      3988
           5       0.21      0.18      0.19      3985

    accuracy                           0.20     20000
   macro avg       0.20      0.20      0.20     20000
weighted avg       0.20      0.20      0.20     20000



The model hasn't learned anything it is just working based on random 1.5 probability, so it's time to check if there is an issue with the correlation of the values in the dataset

In [18]:
print(df.corr()['Performance_Score'].sort_values(ascending=False).round(4))

Performance_Score              1.0000
Sick_Days                      0.0030
Training_Hours                 0.0024
Employee_Satisfaction_Score    0.0017
Projects_Handled               0.0006
Overtime_Hours                -0.0013
Work_Hours_Per_Week           -0.0056
Name: Performance_Score, dtype: float64


as suspected, there is an issue with the dataset, there is a low negligible amount of correlation between the perfomance score value with other columns

I am suspecting it is because of the columns we dropped such as education level, job level but if those were included it would be too hard to make the model generic and try with other environments in a completely different field

So I conclude this model is not reliable and a new model should be trained with a new dataset which has more correlation between performance score with the other metrics of the employee which has actually shows progress